# Find LSSTCamSources in all bands


---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Universite Paris-Saclay
- **Created:** 2026-07-06
- **Last update:** 2026-07-06

## Goal

For each Deep Drilling Field (DDF) and each LSST band, find bright point-like
sources (stars) with `17 <= mag <= 22`, cross-match repeated detections of the
same physical object across visits (same RA/Dec within `MATCH_RADIUS_ARCSEC`),
keep objects with at least `MIN_VISITS_PER_BAND` visits in that band, and
compute the relative flux scatter `sigma_F/F` (expressed in mmag) using
`psfFlux`. Processing is done **band by band, DDF by DDF** to control memory
usage: each chunk is loaded, reduced to per-object statistics, written to
disk, then freed before moving to the next chunk.

Expectation: the relative photometric scatter should be largest in **u** and
**y**, the bands most affected by atmospheric extinction variability
(Rayleigh + aerosols in u, water vapour/aerosols in y) and lower system
throughput.


## Imports

In [ ]:
import gc
import logging
import os
import re
import sys
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.time import Time

from lsst.daf.butler import Butler, Timespan
import lsst.geom as geom
from lsst.geom import SpherePoint, degrees

## Usefull functions

## Logging

In [ ]:
# 1. Recuperer le logger racine (ou creez un logger specifique: logging.getLogger('mon_code'))
log = logging.getLogger()

# 2. Definir le niveau de log global (DEBUG, INFO, WARNING, ERROR)
log.setLevel(logging.INFO)

# 3. Eviter la duplication des handlers si la cellule est executee plusieurs fois
if not log.handlers:
    # 4. Creer un handler qui ecrit vers la sortie standard (capturee par Jupyter)
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)

    # 5. Definir le format des messages (heure, niveau, nom du logger, message)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)

    # 6. Ajouter le handler au logger
    log.addHandler(handler)

# Petit test pour verifier que ca fonctionne
log.info("Le logging est configure et fonctionne dans le notebook !")

## Configuration

**Edit only this cell** to point to the right Butler repository, collections,
input CSV, search radius, and output band.


In [ ]:
# -- Butler --------------------------------------------------------------
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

instrument = "LSSTCam"
skymapName = "lsst_cells_v2"

# All LSST bands to loop over (band-by-band processing to control memory)
BANDS = ["u", "g", "r", "i", "z", "y"]

# -- Input targets ---------------------------------------------------------
target_file = "summary_visit_counts_per_star_V17-21_r2.0deg.csv"

# -- Cross-match search radius ----------------------------------------------
MATCH_RADIUS_ARCSEC = 1.0  # maximum separation for a valid match [arcsec]

# -- Magnitude window for bright stars --------------------------------------
MAG_MIN = 17.0
MAG_MAX = 22.0

# -- Minimum number of visits per band for an object to be kept -------------
MIN_VISITS_PER_BAND = 10

# -- Star/galaxy separation --------------------------------------------------
EXTENDEDNESS_MAX = 0.5  # base_ClassificationExtendedness: 0 = point source, 1 = extended

# -- Flux definition used for the variability analysis -----------------------
FLUX_COL = "psfFlux"
FLUXERR_COL = "psfFluxErr"

# nJy -> AB magnitude zero point (LSST calibrated fluxes are stored in nanojansky)
ABMAG_ZP_NJY = 31.4

# -- Output ------------------------------------------------------------------
OUTPUT_DIR = "output_objectstats"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATE_START = "2025-04-01T00:00:00"
DATE_STOP = "2026-07-01T00:00:00"
time_start = Time(DATE_START, format="isot", scale="utc")
time_stop = Time(DATE_STOP, format="isot", scale="utc")
MJD_START = time_start.mjd
MJD_STOP = time_stop.mjd
DELTAMJD_DAYS = MJD_STOP - MJD_START
log.debug(f"MJD ::: start = {MJD_START} , stop = {MJD_STOP} , delta t = {DELTAMJD_DAYS} days")

SRC_COLUMNS = [
    "coord_ra",
    "coord_dec",
    "parentSourceId",
    "x",
    "y",
    "xErr",
    "yErr",
    "ra",
    "dec",
    "raErr",
    "decErr",
    "calibFlux",
    "calibFluxErr",
    "ap09Flux",
    "ap09FluxErr",
    "ap09Flux_flag",
    "ap12Flux",
    "ap12FluxErr",
    "ap12Flux_flag",
    "ap17Flux",
    "ap17FluxErr",
    "ap17Flux_flag",
    "ap25Flux",
    "ap25FluxErr",
    "ap25Flux_flag",
    "ap35Flux",
    "ap35FluxErr",
    "ap35Flux_flag",
    "sky",
    "skyErr",
    "psfFlux",
    "psfFluxErr",
    "extendedness",
    "sizeExtendedness",
    "apFlux_12_0_flag",
    "apFlux_12_0_instFlux",
    "apFlux_12_0_instFluxErr",
    "apFlux_17_0_flag",
    "apFlux_17_0_instFlux",
    "apFlux_17_0_instFluxErr",
    "apFlux_35_0_flag",
    "apFlux_35_0_instFlux",
    "apFlux_35_0_instFluxErr",
    "extendedness_flag",
    "sizeExtendedness_flag",
    "localBackground_instFlux",
    "localBackground_instFluxErr",
    "localBackground_flag",
    "sky_source",
    "visit",
    "detector",
    "band",
    "physical_filter",
    "sourceId",
]

log.info("Butler configuration done.")

## DDF selected

In [ ]:
# LSST Deep Drilling Fields (RA/Dec J2000)
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "XMM-LSS": (35.7080, -4.750),
    "ECDFS": (53.1250, -27.8),
    "EDFS-a": (58.9, -49.315),
    "EDFS-b": (63.6, -47.6),
    "EDFS": (61.24, -48.423),
    "M49": (187.4, 8.0),
}

## Initialise the Butler

In [ ]:
butler = Butler(repo, collections=collection)
registry = butler.registry
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)
log.info(f"Butler initialised | repo: {repo}")

## Which visits

In [ ]:
visits = butler.registry.queryDimensionRecords("visit")

target_names = sorted({v.target_name for v in visits if v.target_name is not None})
print(target_names)

In [ ]:
ddf_keywords = ["cosmos", "ecdfs", "cdfs", "xmm", "elaiss1", "elais", "edfs"]


def is_ddf(name):
    n = name.lower()
    return any(k in n for k in ddf_keywords)


ddf_names = sorted({name for name in target_names if is_ddf(name)})

for n in ddf_names:
    print(n)

In [ ]:
def normalize_ddf(name):
    n = name.lower()
    if "cosmos" in n:
        return "COSMOS"
    if "ecdfs" in n or "cdfs" in n:
        return "ECDFS"
    if "xmm" in n:
        return "XMM-LSS"
    if "elaiss1" in n or "elais" in n:
        return "ELAIS-S1"
    if "edfs" in n:
        return "EDFS"
    return None


unique_ddf = sorted({normalize_ddf(name) for name in target_names if normalize_ddf(name) is not None})

print(unique_ddf)

In [ ]:
counter = Counter(normalize_ddf(v.target_name) for v in visits if normalize_ddf(v.target_name))

for k, v in counter.items():
    print(k, v)

### Retrieve all Visits in all DDF

In [ ]:
visits = list(butler.registry.queryDimensionRecords("visit"))

ddf_visits = {"COSMOS": [], "XMM-LSS": [], "EDFS": [], "ECDFS": [], "ELAIS-S1": []}

for v in visits:
    name = normalize_ddf(v.target_name)
    if name:
        ddf_visits[name].append(v.id)

# Not working v.band does not exist.
# also index visits by band, needed below to query the source dataset band by band
# visits_band = {v.id: v.band for v in visits}

## Auto-discover the object-table dataset type

The catalogue dataset name changed across pipeline versions:

| Pipeline era | Dataset type name          |
|:-------------|:---------------------------|
| Gen2 / HSC   | `deepCoadd_obj`            |
| DP1          | `objectTable_tract`        |
| DP2+         | `object_table_tract`       |

We probe the registry so the notebook is collection-agnostic. Here we actually
need the **per-visit source table** (not the coadd object table), since we
want repeat single-epoch measurements to build the light curves.


In [ ]:
def dataset_type_exists(butler_obj, name):
    """Return True if `name` is a registered dataset type in this Butler."""
    try:
        butler_obj.registry.getDatasetType(name)
        return True
    except Exception:
        return False


# Prioritised list of candidate per-visit source-table dataset type names
SRC_TABLE_CANDIDATES = [
    "source",
    "sourceTable",
    "sourceTable_visit",  # visit-level source table (fallback)
]

# List all source-table-related types actually in the registry (diagnostic only)
all_src_types = [
    d.name for d in registry.queryDatasetTypes() if "source" in d.name.lower() or "table" in d.name.lower()
]

# Pick the first candidate that is registered
SRC_DATASET = None
for name in SRC_TABLE_CANDIDATES:
    if dataset_type_exists(butler, name):
        SRC_DATASET = name
        log.info(f"Selected src-table dataset type: '{SRC_DATASET}'")
        break

if SRC_DATASET is None:
    raise RuntimeError(
        "No recognised source-table dataset type found in this Butler collection. "
        f"Candidate types seen: {all_src_types}"
    )

In [ ]:
# Quick schema probe on a single DDF/band chunk before running the full loop
probe_refsall = list(butler.query_datasets(SRC_DATASET, where="band = 'y'"))
probe_refs = [r for r in probe_refsall if r.dataId["visit"] in ddf_visits["COSMOS"]]

df_probe = butler.get(probe_refs[0], parameters={"columns": SRC_COLUMNS})
if not isinstance(df_probe, pd.DataFrame):
    df_probe = df_probe.to_pandas()
log.info(f"Probe source table: {len(df_probe)} rows, {len(df_probe.columns)} columns")
log.info(f"Columns of sources dataframe : {df_probe.columns.tolist()}")

# sanity check on the flux unit: LSST calibrated fluxes are in nanojansky (nJy).
# psfFlux for 17-22 mag stars should roughly be in the range 1e3 .. 1e6 nJy.
log.info(f"psfFlux range in probe: {np.nanpercentile(df_probe['psfFlux'], [1, 50, 99])}")

del df_probe, probe_refsall, probe_refs
gc.collect()

## Helper functions: cleaning, magnitudes, object cross-match, per-object statistics

The cross-match strategy ("which sources belong to the same physical star")
is a **friends-of-friends (single-linkage) clustering** in the tangent plane:

1. Project `(ra, dec)` onto a local tangent plane in arcsec:
   `x = ra * cos(dec) * 3600`, `y = dec * 3600` (fine approximation over a
   DDF field of only a few degrees).
2. Build a `scipy.spatial.cKDTree` on `(x, y)` and find all pairs of
   detections closer than `MATCH_RADIUS_ARCSEC` (`tree.query_pairs`).
3. Build a sparse adjacency graph from those pairs and extract its
   **connected components** (`scipy.sparse.csgraph.connected_components`).
   Each connected component = one physical object; all detections in it
   (possibly from many different visits) get the same `object_id`.

This is equivalent to what `astropy.coordinates.match_to_catalog_sky` would
give if you picked one reference visit, but it does not require choosing a
reference catalogue and it is robust to a source being missing in some
visits.


In [ ]:
def clean_sources(df, extendedness_max=EXTENDEDNESS_MAX):
    """Basic quality cuts: remove sky-background sources, flagged
    extendedness/size measurements, non-positive fluxes, and extended
    (galaxy-like) sources. Returns a filtered copy.
    """
    sel = np.ones(len(df), dtype=bool)

    if "sky_source" in df.columns:
        sel &= ~df["sky_source"].to_numpy(dtype=bool)

    if "extendedness_flag" in df.columns:
        sel &= ~df["extendedness_flag"].to_numpy(dtype=bool)

    if "sizeExtendedness_flag" in df.columns:
        sel &= ~df["sizeExtendedness_flag"].to_numpy(dtype=bool)

    if "extendedness" in df.columns:
        ext = df["extendedness"].to_numpy(dtype=float)
        sel &= np.isfinite(ext) & (ext <= extendedness_max)

    flux = df[FLUX_COL].to_numpy(dtype=float)
    fluxerr = df[FLUXERR_COL].to_numpy(dtype=float)
    sel &= np.isfinite(flux) & (flux > 0) & np.isfinite(fluxerr) & (fluxerr > 0)

    return df.loc[sel].reset_index(drop=True)


def add_ab_mag(df, flux_col=FLUX_COL, mag_col="mag", zp=ABMAG_ZP_NJY):
    """Add an AB magnitude column computed from a calibrated flux in nJy."""
    df = df.copy()
    df[mag_col] = -2.5 * np.log10(df[flux_col].to_numpy(dtype=float)) + zp
    return df


def assign_object_ids(df, ra_col="coord_ra", dec_col="coord_dec", radius_arcsec=MATCH_RADIUS_ARCSEC):
    """Group repeated detections of the same physical object using a
    friends-of-friends clustering within `radius_arcsec` on the sky.
    Adds an integer `object_id` column (unique within this dataframe only).
    """
    df = df.copy()
    n = len(df)
    if n == 0:
        df["object_id"] = np.array([], dtype=int)
        return df

    ra = df[ra_col].to_numpy(dtype=float)
    dec = df[dec_col].to_numpy(dtype=float)
    dec_rad = np.deg2rad(dec)

    # local tangent-plane approximation, valid for ~deg-scale DDF fields
    x = ra * np.cos(dec_rad) * 3600.0  # arcsec
    y = dec * 3600.0  # arcsec

    coords = np.column_stack([x, y])
    tree = cKDTree(coords)
    pairs = tree.query_pairs(r=radius_arcsec, output_type="ndarray")

    if len(pairs) > 0:
        row, col = pairs[:, 0], pairs[:, 1]
        graph = coo_matrix((np.ones(len(row)), (row, col)), shape=(n, n))
        n_components, labels = connected_components(graph, directed=False)
    else:
        labels = np.arange(n)
        n_components = n

    df["object_id"] = labels
    log.debug(f"assign_object_ids: {n} detections -> {n_components} distinct objects")
    return df


def summarize_objects(
    df,
    flux_col=FLUX_COL,
    fluxerr_col=FLUXERR_COL,
    mag_col="mag",
    min_visits=MIN_VISITS_PER_BAND,
    mag_min=MAG_MIN,
    mag_max=MAG_MAX,
):
    """Aggregate per-detection rows (already tagged with `object_id`) into
    one row per object with visit count, mean position, mean flux/magnitude,
    and the relative flux scatter sigma_F/F (expressed in mmag too).

    sigma_F/F is computed two ways:
      - `sigmaF_over_F_meas` : *measured* scatter, i.e. std(flux)/mean(flux)
        across the repeated visits (this is what carries any real
        astrophysical/atmospheric variability).
      - `sigmaF_over_F_phot` : *expected* photon-noise-only scatter, i.e. the
        median of fluxErr/flux over the visits (useful to check that the
        measured scatter is not simply dominated by photon noise).
    """
    g = df.groupby("object_id", sort=False)

    n_visits = g.size().rename("n_visits")

    ra_mean = g["coord_ra"].mean().rename("ra")
    dec_mean = g["coord_dec"].mean().rename("dec")

    flux_mean = g[flux_col].mean().rename("flux_mean")
    flux_std = g[flux_col].std(ddof=1).rename("flux_std")

    mag_mean = g[mag_col].mean().rename("mag_mean")

    relerr = df[fluxerr_col] / df[flux_col]
    photnoise_med = relerr.groupby(df["object_id"], sort=False).median().rename("sigmaF_over_F_phot")

    out = pd.concat([n_visits, ra_mean, dec_mean, flux_mean, flux_std, mag_mean, photnoise_med], axis=1)
    out = out.reset_index()

    out["sigmaF_over_F_meas"] = out["flux_std"] / out["flux_mean"]

    mmag_factor = 2.5 / np.log(10.0) * 1000.0  # convert sigma_F/F -> mmag
    out["mmag_meas"] = mmag_factor * out["sigmaF_over_F_meas"]
    out["mmag_phot"] = mmag_factor * out["sigmaF_over_F_phot"]

    sel = (out["n_visits"] >= min_visits) & out["mag_mean"].between(mag_min, mag_max)
    out = out.loc[sel].reset_index(drop=True)

    return out

## Main loop: band by band, DDF by DDF

For each band we first list the source-table refs for that band once, then
loop over DDFs (filtering the refs list by visit id). Each DDF/band chunk is
loaded, cleaned, magnitude-cut, cross-matched into objects, reduced to
per-object statistics, tagged with `ddf`/`band`, and appended to a
per-band accumulator. The raw per-visit dataframe is deleted and
garbage-collected before moving on, so memory stays bounded by the size of
one DDF/band chunk rather than the whole survey.


```memory crash here below: need to obtimize

root INFO: === Band 'u': listing source-table refs ===
root INFO: Band 'u': 1963 refs total (all DDFs)
root INFO:   [u/COSMOS] 10181642 raw rows -> 1296916 after mag/quality cuts -> 19088 objects with >= 10 visits
root INFO:   [u/ECDFS] 735416 raw rows -> 82412 after mag/quality cuts -> 824 objects with >= 10 visits
root INFO:   [u/EDFS] 1016779 raw rows -> 152924 after mag/quality cuts -> 147 objects with >= 10 visits
root INFO:   [u/ELAIS-S1] 5156250 raw rows -> 557949 after mag/quality cuts -> 14117 objects with >= 10 visits
root INFO:   [u/XMM-LSS] 3248335 raw rows -> 355355 after mag/quality cuts -> 12051 objects with >= 10 visits
root INFO: Band 'u': 46227 objects (all DDFs) written to output_objectstats/objectstats_band_u.parquet
root INFO: === Band 'g': listing source-table refs ===
root INFO: Band 'g': 4158 refs total (all DDFs)

```

In [ ]:
all_band_results = {}

for band in BANDS:
    log.info(f"=== Band '{band}': listing source-table refs ===")
    refsall = list(butler.query_datasets(SRC_DATASET, where="band = :b", bind={"b": band}))
    log.info(f"Band '{band}': {len(refsall)} refs total (all DDFs)")

    band_chunks = []

    for ddf_name in unique_ddf:
        visit_ids = set(ddf_visits[ddf_name])
        refs = [r for r in refsall if r.dataId["visit"] in visit_ids]
        if not refs:
            log.info(f"  [{band}/{ddf_name}] no refs, skipping")
            continue

        dfs = []
        for ref in refs:
            d = butler.get(ref, parameters={"columns": SRC_COLUMNS})
            if not isinstance(d, pd.DataFrame):
                d = d.to_pandas()
            dfs.append(d)
        df = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]
        del dfs

        n_raw = len(df)
        df = clean_sources(df)
        df = add_ab_mag(df)

        # pre-cut on magnitude BEFORE cross-matching: fewer rows -> faster KDTree
        df = df.loc[df["mag"].between(MAG_MIN, MAG_MAX)].reset_index(drop=True)

        if len(df) == 0:
            log.info(f"  [{band}/{ddf_name}] {n_raw} raw rows -> 0 after cuts, skipping")
            del df
            gc.collect()
            continue

        df = assign_object_ids(df)
        obj = summarize_objects(df)
        obj["ddf"] = ddf_name
        obj["band"] = band

        log.info(
            f"  [{band}/{ddf_name}] {n_raw} raw rows -> {len(df)} after mag/quality cuts "
            f"-> {len(obj)} objects with >= {MIN_VISITS_PER_BAND} visits"
        )

        band_chunks.append(obj)

        del df, obj
        gc.collect()

    if band_chunks:
        df_band = pd.concat(band_chunks, ignore_index=True)
    else:
        df_band = pd.DataFrame()

    out_path = os.path.join(OUTPUT_DIR, f"objectstats_band_{band}.parquet")
    df_band.to_parquet(out_path, index=False)
    log.info(f"Band '{band}': {len(df_band)} objects (all DDFs) written to {out_path}")

    all_band_results[band] = df_band

    del refsall, band_chunks, df_band
    gc.collect()

## Combine all bands and inspect the results

In [ ]:
df_all = pd.concat(
    [df for df in all_band_results.values() if len(df) > 0],
    ignore_index=True,
)
log.info(f"Total objects across all bands/DDFs: {len(df_all)}")

df_all.to_csv(os.path.join(OUTPUT_DIR, "objectstats_allbands.csv"), index=False)
df_all.head()

In [ ]:
df_all.groupby("band")["mmag_meas"].describe()[["count", "mean", "50%", "std"]]

## Plot: relative photometric scatter (mmag) per band

Boxplot of `mmag_meas` (measured scatter, using `psfFlux`) grouped by band,
in the standard LSST band order `u, g, r, i, z, y`. We expect the largest
scatter in **u** and **y**.


In [ ]:
band_order = [b for b in ["u", "g", "r", "i", "z", "y"] if b in df_all["band"].unique()]

data = [df_all.loc[df_all["band"] == b, "mmag_meas"].dropna().to_numpy() for b in band_order]

fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot(data, labels=band_order, showfliers=False)
ax.set_xlabel("Band")
ax.set_ylabel(r"$\sigma_F/F$ (mmag)")
ax.set_title(
    f"Relative PSF-flux scatter, {MAG_MIN:.0f} < mag < {MAG_MAX:.0f}, "
    f">= {MIN_VISITS_PER_BAND} visits/band"
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "mmag_scatter_per_band.png"), dpi=150)
plt.savefig(os.path.join(OUTPUT_DIR, "mmag_scatter_per_band.pdf"))
plt.show()

In [ ]:
# Cross-check: measured scatter vs. photon-noise-only expectation, per band
fig, ax = plt.subplots(figsize=(6, 6))
for b in band_order:
    sub = df_all.loc[df_all["band"] == b]
    ax.scatter(sub["mag_mean"], sub["mmag_meas"], s=4, alpha=0.4, label=b)
ax.set_xlabel("mean magnitude")
ax.set_ylabel(r"$\sigma_F/F$ (mmag), measured")
ax.set_yscale("log")
ax.legend(markerscale=3, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()